<a href="https://colab.research.google.com/github/albhoe/593Project/blob/main/testingproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade torchao

In [ ]:
import pandas as pd
import io
from google.colab import drive
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader
from datasets import Dataset

drive.mount('/content/drive')

In [ ]:
file_path_drive = '/content/drive/MyDrive/593project/ao3_14400001-14500000.jsonl'
generation_path_drive = '/content/drive/MyDrive/593project/fine_tuned_bart_lora_generation_saved'
classification_path_drive = '/content/drive/MyDrive/593project/fine_tuned_bart_lora_classification_saved'

line_count = 0
df = pd.DataFrame()

with open(file_path_drive, 'r', encoding='utf-8') as f:
    for line in f:
        if line_count >= 2:
            break
        line_count += 1
        # Use io.StringIO to pass the literal JSON string safely
        df = pd.concat([df, pd.read_json(io.StringIO(line), lines=True)], ignore_index=True)
metadata_df = pd.json_normalize(df['metadata'])
df = df.drop(columns=['metadata','id'])
df = pd.concat([df.reset_index(drop=True), metadata_df.reset_index(drop=True)], axis=1)
display(df.head())
df = df[['text','Additional Tags','Rating']]

print("\nFirst 5 rows of data from Google Drive:")
display(df.head())

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large')

# Load the generation model
print(f"Loading generation model from {generation_path_drive}...")
config_generation = PeftConfig.from_pretrained(generation_path_drive)
base_model_generation = AutoModelForCausalLM.from_pretrained(config_generation.base_model_name_or_path)
generation_model = PeftModel.from_pretrained(base_model_generation, generation_path_drive)
print("Generation model loaded successfully.")

# Load the classification model
print(f"Loading classification model from {classification_path_drive}...")
config_classification = PeftConfig.from_pretrained(classification_path_drive)
base_model_classification = AutoModelForSequenceClassification.from_pretrained(config_classification.base_model_name_or_path, num_labels=2)
classification_model = PeftModel.from_pretrained(base_model_classification, classification_path_drive)
print("Classification model loaded successfully.")

In [ ]:
# Example of making a single prediction with the generation model

def generate_text(model, tokenizer, prompt_text, max_length=50):
    model.eval() # Ensure the model is in evaluation mode
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Encode the input prompt, truncating if too long
    input_ids = tokenizer.encode(prompt_text, return_tensors='pt', truncation=True).to(device)
    current_input_length = input_ids.shape[1]

    # Calculate remaining capacity for new tokens
    max_model_length = tokenizer.model_max_length # Get the model's max input length, usually 1024 for BART
    remaining_capacity = max_model_length - current_input_length

    # Ensure max_new_tokens does not cause the total sequence to exceed max_model_length
    # Also, ensure at least 1 token is generated if possible
    max_new_tokens_to_generate = max(1, min(max_length, remaining_capacity))

    # Generate text
    # You can customize generation parameters like num_beams, do_sample, top_k, top_p, etc.
    output_sequences = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens_to_generate, # Use max_new_tokens to control the number of newly generated tokens
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        num_return_sequences=1
    )

    # Decode the generated text
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return generated_text

# Apply the generation process to each row in the DataFrame
df['generated_tags'] = df['text'].apply(lambda x: generate_text(generation_model, tokenizer, x))

print("DataFrame with generated tags:")
display(df.head())

In [ ]:
def predict_classification_label(model, tokenizer, text):
    model.eval() # Ensure the model is in evaluation mode
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Encode the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits

    # Apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # Get the predicted label (index of the max probability)
    predicted_label = torch.argmax(probabilities, dim=1).item()

    rating_map = {
        0: 'Not Rated',
        1: 'General Audiences',
        2: 'Teen And Up Audiences',
        3: 'Mature',
        4: 'Explicit'
    }

    return rating_map.get(predicted_label, 'Unknown')

# Apply the classification process to each row in the DataFrame
# Assuming 'Rating' is a binary classification, e.g., 0 for 'General Audiences' and 1 for 'Mature/Explicit'
# You might need to map these labels based on your actual data and model training.
# For now, let's assume the model outputs 0 or 1.
df['predicted_rating'] = df['text'].apply(lambda x: predict_classification_label(classification_model, tokenizer, x))

print("DataFrame with predicted ratings:")
display(df.head())